In [ ]:
import os
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
import kagglehub
kagglehub.login()

path = kagglehub.competition_download(
    'ml-intensive-yandex-academy-spring-2026'
)

BASE = path + "/dataset"

train_dir = BASE + "/train_images"
test_dir = BASE + "/test_images"
labels_path = BASE + "/train_solution.csv"

df = pd.read_csv(labels_path)
df.head()

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
df = pd.read_csv(labels_path)
df.columns = ['Id', 'target_feature']

In [ ]:
TARGET_COL = None

for col in df.columns:
    if col.lower() in ['target_feature', 'target', 'label']:
        TARGET_COL = col

print("Target column:", TARGET_COL)

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[TARGET_COL],
    random_state=42
)

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.img_dir, f"{row['Id']}.jpg")
        image = Image.open(img_path).convert("RGB")

        if self.transforms:
            image = self.transforms(image)

        label = torch.tensor(row['target_feature'], dtype=torch.float32)

        return image, label

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.GaussianBlur(3),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3)
])

val_tfms = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor()
])

In [ ]:
train_ds = FaceDataset(train_df, train_dir, train_tfms)
val_ds   = FaceDataset(val_df, train_dir, val_tfms)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(c, c//16),
            nn.ReLU(),
            nn.Linear(c//16, c),
            nn.Sigmoid()
        )

    def forward(self, x):
        b,c,h,w = x.shape
        y = x.view(b,c,-1).mean(2)
        y = self.fc(y).view(b,c,1,1)
        return x * y


class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_c)

        self.se = SEBlock(out_c)

        self.skip = nn.Sequential()
        if in_c != out_c or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        identity = self.skip(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = self.se(out)
        out += identity

        return F.relu(out)


class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3,32,3,2,1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.layer1 = self.block(32,64)
        self.layer2 = self.block(64,128)
        self.layer3 = self.block(128,256)
        self.layer4 = self.block(256,512)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512,128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,1)
        )

    def block(self, in_c, out_c):
        return nn.Sequential(
            ResBlock(in_c, out_c, stride=2),
            ResBlock(out_c, out_c)
        )

    def forward(self,x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.fc(x)

In [ ]:
pos_weight = torch.tensor([3.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
model = Model().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [ ]:
def train_epoch(model, loader, threshold=0.5):
    model.train()

    losses = []
    preds, targets = [], []

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device).unsqueeze(1)

        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds.extend(probs)
        targets.extend(y.cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    preds_bin = (preds > threshold).astype(int)

    f1 = f1_score(targets, preds_bin)
    recall = recall_score(targets, preds_bin)

    return np.mean(losses), f1, recall


def evaluate(model, loader):
    model.eval()

    preds, targets = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)

            probs = torch.sigmoid(logits).cpu().numpy()
            preds.extend(probs)
            targets.extend(y.numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    best_f1, best_t = 0, 0

    for t in np.linspace(0.1, 0.9, 30):
        p = (preds > t).astype(int)
        f1 = f1_score(targets, p)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    recall = recall_score(targets, (preds > best_t).astype(int))

    return best_f1, recall, best_t

In [ ]:
def plot_metrics(train_losses, train_f1s, val_f1s, val_recalls):
    clear_output(wait=True)

    plt.figure(figsize=(15,5))

    # Loss
    plt.subplot(1,3,1)
    plt.plot(train_losses, label="Train Loss")
    plt.title("Loss")
    plt.legend()

    # F1
    plt.subplot(1,3,2)
    plt.plot(train_f1s, label="Train F1")
    plt.plot(val_f1s, label="Val F1")
    plt.title("F1 Score")
    plt.legend()

    # Recall
    plt.subplot(1,3,3)
    plt.plot(val_recalls, label="Val Recall")
    plt.title("Recall")
    plt.legend()

    plt.show()

In [ ]:
EPOCHS = 50

train_losses = []
train_f1s = []
val_f1s = []
val_recalls = []

best_threshold = 0.5

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}")

    loss, train_f1, train_recall = train_epoch(
        model, train_loader, threshold=best_threshold
    )

    val_f1, val_recall, best_threshold = evaluate(model, val_loader)

    scheduler.step()

    train_losses.append(loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)
    val_recalls.append(val_recall)

    print(f"Loss: {loss:.4f}")
    print(f"Train F1: {train_f1:.4f} | Train Recall: {train_recall:.4f}")
    print(f"Val F1: {val_f1:.4f} | Val Recall: {val_recall:.4f}")
    print(f"Best Threshold: {best_threshold:.2f}")

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_threshold': best_threshold,
        'train_losses': train_losses,
        'train_f1s': train_f1s,
        'val_f1s': val_f1s,
        'val_recalls': val_recalls
    }

    torch.save(checkpoint, 'checkpoint.pth')

    plot_metrics(train_losses, train_f1s, val_f1s, val_recalls)


plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.legend()
plt.title("Loss")
plt.show()

plt.figure()
plt.plot(train_f1s, label="Train F1")
plt.plot(val_f1s, label="Val F1")
plt.legend()
plt.title("F1 Score")
plt.show()

plt.figure()
plt.plot(val_recalls, label="Val Recall")
plt.legend()
plt.title("Recall")
plt.show()

In [ ]:
import torch

checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_threshold': best_threshold,
    'train_losses': train_losses,
    'train_f1s': train_f1s,
    'val_f1s': val_f1s,
    'val_recalls': val_recalls
}

torch.save(checkpoint, 'checkpoint.pth')

In [ ]:
plt.plot(train_losses, label="loss")
plt.legend()
plt.show()

plt.plot(val_f1s, label="f1")
plt.legend()
plt.show()

In [ ]:
class TestDataset(Dataset):
    def __init__(self, img_dir, transforms):
        self.img_dir = img_dir
        self.files = sorted(os.listdir(img_dir))
        self.transforms = transforms

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")

        img = self.transforms(img)
        return img, img_name

In [ ]:
test_ds = TestDataset(test_dir, val_tfms)
test_loader = DataLoader(test_ds, batch_size=128)

In [ ]:
model.eval()
preds = []

with torch.no_grad():
    for x, names in tqdm(test_loader):
        x = x.to(device)
        logits = model(x)

        probs = torch.sigmoid(logits).cpu().numpy()
        preds.extend(probs)

In [ ]:
threshold = best_threshold

final_preds = (np.array(preds) > threshold).astype(int)

ids = [int(name.split(".")[0]) for name in test_ds.files]

sub = pd.DataFrame({
    "Id": ids,
    "target_feature": final_preds.flatten()
})

sub = sub.sort_values("Id")

sub.to_csv("submission.csv", index=False)